# HU-04 · Ranking mensual

[Inicio](../../tests.ipynb) · Anterior: [HU-03 · Carrera multijugador](hu03_carrera.ipynb) · Siguiente: [HU-05 · Analítica personal](hu05_analitica.ipynb)

Las fichas incluyen los criterios del Plan Maestro. Las referencias U/G/F/W/D y las rutas abreviadas en **Evidencia** se interpretan en la [matriz maestra](../../docs/test/qa-master-matrix.md). Una ejecución local no implica un resultado publicado ni prueba un control de entrega.

## TC-FUNC-10

- **Resumen de la ficha:** Cambiar filtros válidos de dificultad e idioma, actualizar las filas y conservar el documento sin recarga.
- **Método previsto:** Manual + E2E.
- **Estado:** **V — Verificado**.
- **Evidencia:** En la matriz (**P**): Filtros y cancelación en `practice.test.mjs` (**F**); filtros en `test_practice_matches.py` y `integration/test_practice_postgresql.py` (**U**). Falta verificar actualización sin recarga en navegador.
  
  Evidencia local posterior o adicional: Chromium visible verificó filas al cambiar dificultad e idioma, marcador del mismo documento y cero solicitudes de navegación (Engram #990). Una ejecución local posterior aprobó ambos escenarios de ranking, 21 pasos. [Fuente de prueba](../../frontend/e2e/ranking.feature).
- **Brecha o límite:** No se infiere una ejecución verde en `main`.

**Criterio de aceptación literal** · Plan §5.5 → [HU-04 #11](https://github.com/1Bnja/Typr/issues/11).

```gherkin
Característica: Consulta del ranking global

  Escenario: Filtrado dinámico sin recarga completa
    Dado que el usuario visualiza la tabla del ranking global
    Cuando selecciona un nuevo valor para los filtros de idioma o dificultad
    Entonces la tabla se actualiza de forma asíncrona con los nuevos datos sin recargar la página completa
```

**Gherkin ejecutable (original en inglés)** · [Fuente](../../frontend/e2e/ranking.feature).

```gherkin
Feature: Monthly ranking filters

  @TC-FUNC-10
  Scenario: Changing difficulty and language replaces the ranking in the same document
    Given distinct practice results for Easy Spanish, Medium Spanish and Medium English
    And I register a new account in the browser
    When I open the monthly ranking for the seeded month
    Then only "rank_easy_es" appears in the ranking
    When I select the "Medium" ranking difficulty
    Then only "rank_medium_es" appears in the ranking
    When I select the "English" ranking language
    Then only "rank_medium_en" appears in the ranking
    And the ranking remains in the original document without a navigation request
```

## TC-FUNC-11

- **Resumen de la ficha:** Excluir partidas por debajo de 75 % e incluir las de exactamente 75 %.
- **Método previsto:** API.
- **Estado:** **V — Verificado**.
- **Evidencia:** En la matriz (**V**): Ranking excluye por debajo de 75 % e incluye exactamente 75 %: `test_practice_matches.py::test_monthly_ranking_excludes_matches_below_75_precision` (**U**). [Fuente de prueba](../../backend/tests/test_practice_matches.py). Véase [RI-12](../integridad/ciclo_metricas_sesion_y_textos.ipynb#RI-12); comparten ejecución.
- **Brecha o límite:** No se identificó una brecha para el escenario y entorno documentados.

**Criterio de aceptación literal** · Plan §5.5 → [HU-04 #11](https://github.com/1Bnja/Typr/issues/11).

```gherkin
Característica: Consulta del ranking global

  Escenario: Exclusión de partidas con precisión insuficiente
    Dado que existen partidas registradas en el sistema
    Cuando el sistema computa las posiciones para el ranking global
    Entonces omite del listado todas las partidas cuya precisión sea menor al 75%
```

No se identificó un `.feature` para el umbral: la comprobación es de API/integración.

## TC-FUNC-12

- **Resumen de la ficha:** Una combinación válida sin resultados muestra el estado vacío y no un error ni una tabla.
- **Método previsto:** Manual.
- **Estado:** **V — Verificado**.
- **Evidencia:** En la matriz (**P**): Solo estático: `frontend/src/features/practice/pages/leaderboard.jsx` distingue vacío y error. Falta probar un filtro sin resultados y observar el estado vacío en navegador.
  
  Evidencia local posterior o adicional: Primero hubo HTTP 200 con lista vacía y sin tabla ni alerta, pero falló el texto exacto por aparecer en inglés (Engram #990). La corrida local `typr-ranking-copy.xUlzkH` aprobó ambos escenarios y 21 pasos; el resultado incluye «Aún no hay partidas para esta categoría». [Fuente de prueba](../../frontend/e2e/ranking.feature).
- **Brecha o límite:** No se infiere una ejecución verde en `main`.

**Criterio de aceptación literal** · Plan §5.5 → [HU-04 #11](https://github.com/1Bnja/Typr/issues/11).

```gherkin
Característica: Consulta del ranking global

  Escenario: Filtro sin registros disponibles
    Dado que el usuario aplica una combinación de filtros sin partidas válidas
    Cuando la consulta al ranking finaliza
    Entonces la vista muestra el mensaje "Aún no hay partidas para esta categoría"
```

**Gherkin ejecutable (original en inglés)** · [Fuente](../../frontend/e2e/ranking.feature).

```gherkin
Feature: Monthly ranking filters

  @TC-FUNC-12
  Scenario: A valid empty category displays the required empty message
    Given a completed Hard Spanish result but no Hard English result
    And I register a new account in the browser
    When I open the monthly ranking for the seeded month
    And I select the "Hard" ranking difficulty
    Then only "rank_hard_es" appears in the ranking
    When I select the "English" ranking language
    Then the successful ranking response is empty and no table or error is displayed
    And the empty ranking message is exactly "Aún no hay partidas para esta categoría"
```

## Reproducción local opcional

Prepare el entorno desechable de la [guía de pruebas](../../test/README.md). Ejecute solo la celda que necesita, con el kernel iniciado en la raíz de Typr o dentro de `notebooks/`. No ejecute en paralelo. Abrir el cuaderno no inicia pruebas; la salida describe una corrida local, no cambia el estado de las fichas. Chromium se abre en una sesión gráfica local. Las celdas de backend ejecutan la suite indicada completa, no solo los casos de esta página.

### Criterios de aceptación · TC-FUNC-10 y TC-FUNC-12

**Plan (§5.5):** Los filtros actualizan el ranking sin recarga completa; una categoría válida sin partidas muestra un estado vacío.

**Qué comprueba esta celda:** Chromium filtra ambos casos. Al cambiar dificultad e idioma cambian las filas esperadas, se conserva el documento original y no hay solicitud de navegación. La categoría vacía devuelve HTTP 200 con lista vacía y muestra «Aún no hay partidas para esta categoría», sin tabla ni error.

**Límite:** No comprueba el umbral de precisión; corresponde a la siguiente celda.

In [ ]:
from pathlib import Path
from runpy import run_path

directorio = Path.cwd().resolve()
for raiz in (directorio, *directorio.parents):
    if (raiz / "docker-compose.test.yml").is_file() and (raiz / "notebooks/ejecutar_qa.py").is_file():
        break
else:
    raise RuntimeError("Inicie el kernel desde la raíz de Typr o dentro de notebooks/.")

run_path(str(raiz / "notebooks/ejecutar_qa.py"))["ejecutar"]('browser-gherkin', etiquetas='@TC-FUNC-10 or @TC-FUNC-12')

### Criterios de aceptación · TC-FUNC-11 y RI-12

**Plan (§§5.5–5.6):** El ranking excluye partidas por debajo del 75 % de precisión; el 75 % exacto no queda excluido por ese umbral.

**Qué comprueba esta celda:** Ejecuta toda la suite de pruebas unitarias, integración y migraciones del backend. El umbral se comprueba en [test_monthly_ranking_excludes_matches_below_75_precision](../../backend/tests/test_practice_matches.py): excluye resultados de 74 % y 74,9 % y admite resultados de 75 % y 76 %. `test_practice_postgresql.py` comprueba el ranking filtrado en PostgreSQL, no el límite de precisión.

**Límite:** La aserción es de API y PostgreSQL, no de interfaz. TC-FUNC-11 y RI-12 comparten esta comprobación.

In [ ]:
from pathlib import Path
from runpy import run_path

directorio = Path.cwd().resolve()
for raiz in (directorio, *directorio.parents):
    if (raiz / "docker-compose.test.yml").is_file() and (raiz / "notebooks/ejecutar_qa.py").is_file():
        break
else:
    raise RuntimeError("Inicie el kernel desde la raíz de Typr o dentro de notebooks/.")

run_path(str(raiz / "notebooks/ejecutar_qa.py"))["ejecutar"]('backend-unit-integration-migrations')

[Inicio](../../tests.ipynb) · Anterior: [HU-03 · Carrera multijugador](hu03_carrera.ipynb) · Siguiente: [HU-05 · Analítica personal](hu05_analitica.ipynb)